In [23]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import ast
from difflib import get_close_matches
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [24]:
# 2. Load the New TMDB Movie Dataset
df = pd.read_csv("data/tmdb_movie_dataset.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df[["tmdbId", "title", "release_date", "vote_average"]].head(3))


Dataset shape: (4602, 21)
Columns: ['budget', 'genres', 'homepage', 'tmdbId', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'ratingId']


,tmdbId,title,release_date,vote_average
0,5,Four Rooms,1995-12-09,6.5
1,11,Star Wars,1977-05-25,8.1
2,12,Finding Nemo,2003-05-30,7.6


In [25]:
# 3. Data Pre-processing and Feature Extraction

def extract_names(text):
    try:
        items = ast.literal_eval(text)
        if isinstance(items, list):
            return " ".join(
                str(item.get("name", ""))
                for item in items
                if isinstance(item, dict)
            )
    except (ValueError, SyntaxError, TypeError):
        pass
    return ""


# Convert genres and keywords into clean text features.
df["genres_clean"] = df["genres"].fillna("").apply(extract_names)
df["keywords_clean"] = df["keywords"].fillna("").apply(extract_names)

# Convert genres and keywords into sets for hard filtering.
df["genre_set"] = df["genres_clean"].apply(
    lambda x: set(x.split()) if x else set()
)
df["keyword_set"] = df["keywords_clean"].apply(
    lambda x: set(x.split()) if x else set()
)

# Keep only movies with a title and reset the row index.
df = df[df["title"].notna()].copy().reset_index(drop=True)

display(df[["title", "genres_clean", "keywords_clean"]].head(3))


,title,genres_clean,keywords_clean
0,Four Rooms,Crime Comedy,hotel new year's eve witch bet hotel room sper...
1,Star Wars,Adventure Action Science Fiction,android galaxy hermit death star lightsaber je...
2,Finding Nemo,Animation Family,father son relationship harbor underwater fish...


In [26]:
# 4. Create Feature Vectors and Cosine Similarity

# We use simple binary feature vectors instead of TF-IDF.
# Features are based only on genres and keywords.
df["content"] = (
    df["genres_clean"] + " " + df["keywords_clean"]
).str.strip()

vectorizer = CountVectorizer(
    binary=True,
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b"
)

feature_matrix = vectorizer.fit_transform(df["content"])
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)

# Map each movie title to its row index.
# Duplicate titles are handled by keeping the first occurrence.
indices = pd.Series(
    df.index,
    index=df["title"].str.strip().str.lower()
).drop_duplicates()

print("Feature Matrix size:", feature_matrix.shape)
print("Cosine Similarity Matrix size:", cosine_sim.shape)


Feature Matrix size: (4602, 7128)
Cosine Similarity Matrix size: (4602, 4602)


In [30]:
# 5. Content-Based Recommendation Function

GENRE_OVERLAP_THRESHOLD = 0.50
MIN_COMMON_KEYWORDS = 2


def genre_overlap_ratio(query_genres, candidate_genres):
    if not query_genres:
        return 0.0
    return len(query_genres & candidate_genres) / len(query_genres)


def find_movie_title(title):
    query = str(title).strip().lower()

    if not query:
        return None

    # Exact match
    if query in indices.index:
        return df.loc[indices[query], "title"]

    # Case-insensitive partial match
    partial_matches = [
        t for t in df["title"].dropna().unique()
        if query in str(t).lower()
    ]

    if partial_matches:
        return partial_matches[0]

    # Fuzzy match for small spelling differences
    title_map = {
        str(t).lower(): str(t)
        for t in df["title"].dropna().unique()
    }

    close = get_close_matches(
        query,
        list(title_map.keys()),
        n=1,
        cutoff=0.65
    )

    if close:
        return title_map[close[0]]

    return None


def recommend(title, top_n=10):
    # Recommend movies using genre/keyword filters, then cosine similarity.
    matched_title = find_movie_title(title)

    if matched_title is None:
        return None

    idx = indices[matched_title.lower()]

    query_genres = df.loc[idx, "genre_set"]
    query_keywords = df.loc[idx, "keyword_set"]

    if not query_genres:
        return None

    # ---------- Stage 1: Genre Hard Filter ----------
    # Candidate must share at least 50% of the input movie's genres.
    genre_candidates = []

    for i, row in df.iterrows():
        if i == idx:
            continue

        genre_ratio = genre_overlap_ratio(
            query_genres,
            row["genre_set"]
        )

        if genre_ratio >= GENRE_OVERLAP_THRESHOLD:
            genre_candidates.append(i)

    # ---------- Stage 2: Keyword Hard Filter ----------
    # Candidate must also share at least 2 keywords with the input movie.
    qualified_candidates = []

    for i in genre_candidates:
        common_keywords = query_keywords & df.loc[i, "keyword_set"]

        if len(common_keywords) >= MIN_COMMON_KEYWORDS:
            qualified_candidates.append(i)

    if not qualified_candidates:
        return None

    # ---------- Stage 3: Cosine Similarity Ranking ----------
    # Cosine similarity is used only to rank already-qualified movies.
    ranked_candidates = sorted(
        qualified_candidates,
        key=lambda i: cosine_sim[idx][i],
        reverse=True
    )[:top_n]

    result = df[
        ["title", "genres_clean", "keywords_clean"]
    ].iloc[ranked_candidates].copy()

    result.insert(0, "rank", range(1, len(result) + 1))

    result["genre_overlap_ratio"] = [
        round(
            genre_overlap_ratio(
                query_genres,
                df.loc[i, "genre_set"]
            ),
            2
        )
        for i in ranked_candidates
    ]

    result["common_keyword_count"] = [
        len(query_keywords & df.loc[i, "keyword_set"])
        for i in ranked_candidates
    ]

    result["similarity_score"] = [
        round(cosine_sim[idx][i], 4)
        for i in ranked_candidates
    ]

    result["keywords_clean"] = result["keywords_clean"].apply(
        lambda x: ", ".join(x.split()[:5]) if x else ""
    )

    result = result.rename(columns={
        "title": "movie_title",
        "genres_clean": "genres",
        "keywords_clean": "keywords"
    })

    return result.reset_index(drop=True)


In [32]:
# 6. Test the Recommendation Function

recommend("Avarta", top_n=10)


,rank,movie_title,genres,keywords,genre_overlap_ratio,common_keyword_count,similarity_score
0,1,Soldier,Action War Science Fiction,"space, marine, dystopia, alien, planet",0.6,4,0.4332
1,2,Jupiter Ascending,Science Fiction Fantasy Action Adventure,"jupiter, space, woman, director, 3d",1.0,3,0.4148
2,3,Star Trek Into Darkness,Action Adventure Science Fiction,"spacecraft, friendship, sequel, futuristic, space",0.8,4,0.3710
3,4,A Sound of Thunder,Thriller Science Fiction Adventure Action,"dying, and, death, time, travel",0.8,3,0.3629
4,5,Planet of the Apes,Thriller Science Fiction Action Adventure,"gorilla, space, marine, space, suit",0.8,5,0.3614
5,6,Aliens in the Attic,Adventure Comedy Family Fantasy Science Fiction,"alien, comedy, duringcreditsstinger, beforecre...",0.8,2,0.3487
6,7,John Carter,Action Adventure Science Fiction,"based, on, novel, mars, medallion",0.8,6,0.3335
7,8,Predator,Science Fiction Action Adventure Thriller,"central, and, south, america, predator",0.8,3,0.3246
8,9,AVP: Alien vs. Predator,Adventure Science Fiction Action,"saving, the, world, predator, laserpointer",0.8,3,0.3246
9,10,Independence Day,Action Adventure Science Fiction,"spacecraft, patriotism, countdown, independenc...",0.8,3,0.3246


In [33]:
# 7. User Input - Enter a Movie Title to Get Recommendations

favorite_movie = input(
    "Please enter a movie title (e.g., Avatar): "
).strip()

matched_title = find_movie_title(favorite_movie)

if matched_title is None:
    print("\nMovie not found in the dataset.")
    print("Please try another movie title.")
else:
    print(f"\nSelected Movie: {matched_title}")
    recommendations = recommend(matched_title, top_n=10)

    if recommendations is None or recommendations.empty:
        print(
            "\nNo recommendations meet the required criteria "
            "(at least 50% genre overlap and at least 2 common keywords)."
        )
    else:
        print("\nTop Content-Based Recommendations:")
        display(recommendations)



Selected Movie: Toy Story

Top Content-Based Recommendations:


,rank,movie_title,genres,keywords,genre_overlap_ratio,common_keyword_count,similarity_score
0,1,Toy Story 2,Animation Comedy Family,"museum, prosecution, identity, crisis, airplane",1.00,5,0.4051
1,2,Toy Story 3,Animation Family Comedy,"hostage, college, toy, barbie, animation",1.00,4,0.3944
2,3,Pinocchio,Animation Family,"italy, lie, magic, fairy, pinocchio",0.67,5,0.3944
3,4,Drive Me Crazy,Drama Comedy Romance Family,"high, school, prom, next, door",0.67,2,0.3266
4,5,The Brave Little Toaster,Fantasy Adventure Animation Comedy Family Music,"growing, up, coming, of, age",1.00,2,0.2962
5,6,The Flintstones,Fantasy Comedy Family,"manager, jealousy, bad, mother-in-law, adoption",0.67,3,0.2887
6,7,Alvin and the Chipmunks,Comedy Music Family Fantasy Animation,"pop, pop, star, record, producer",1.00,2,0.2752
7,8,Down to You,Comedy Drama Family Romance,"lovesickness, new, love, love, of",0.67,2,0.2667
8,9,Coraline,Animation Family,"dream, eye, stuffed, animal, parallel",0.67,2,0.2667
9,10,Hotel Transylvania 2,Animation Comedy Family,"transylvania, hotel, witch, technology, magic",1.00,2,0.2635


In [34]:
# 8. Evaluation: Genre Precision@10 + Average Genre Overlap + Keyword Match

def genre_precision_at_k(title, k=10):
    # Share of top-k recommendations that share at least one genre.
    matched_title = find_movie_title(title)

    if matched_title is None:
        return None

    selected_genres = df.loc[
        indices[matched_title.lower()],
        "genre_set"
    ]

    if not selected_genres:
        return None

    recommended = recommend(matched_title, k)

    if recommended is None or recommended.empty:
        return None

    matches = recommended["genre_overlap_ratio"] > 0
    return matches.mean()


import random
random.seed(42)

valid_titles = (
    df[df["genres_clean"].str.strip() != ""]["title"]
    .drop_duplicates()
    .tolist()
)

sample_size = min(100, len(valid_titles))
sample_titles = random.sample(valid_titles, sample_size)

precisions = []
overlap_ratios = []
keyword_matches = []

for title in sample_titles:
    recommended = recommend(title, top_n=10)

    if recommended is None or recommended.empty:
        continue

    precision = genre_precision_at_k(title, k=10)

    if precision is not None:
        precisions.append(precision)

    overlap_ratios.append(
        recommended["genre_overlap_ratio"].mean()
    )

    keyword_matches.append(
        recommended["common_keyword_count"].mean()
    )

if precisions:
    print(
        f"Average Genre Precision@10 "
        f"(n={len(precisions)}): {sum(precisions) / len(precisions):.4f}"
    )

if overlap_ratios:
    print(
        f"Average Genre Overlap Ratio "
        f"(n={len(overlap_ratios)}): "
        f"{sum(overlap_ratios) / len(overlap_ratios):.4f}"
    )

if keyword_matches:
    print(
        f"Average Common Keywords per Recommendation "
        f"(n={len(keyword_matches)}): "
        f"{sum(keyword_matches) / len(keyword_matches):.2f}"
    )


Average Genre Precision@10 (n=91): 1.0000
Average Genre Overlap Ratio (n=91): 0.7990
Average Common Keywords per Recommendation (n=91): 2.92
